# 🔧 Notebook 2: Naive Approach

Single server orchestration with manual state management.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to add state persistence
- Manual retry logic
- Compensation (rollback) actions
- Why this approach becomes unmaintainable

In [ ]:
import psycopg2
import json
import uuid
import time
import random
from datetime import datetime
from enum import Enum

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="temporal", user="postgres", password="postgres"
)
conn.autocommit = True

cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS order_state (
        id VARCHAR(36) PRIMARY KEY,
        status VARCHAR(50) NOT NULL,
        current_step VARCHAR(50),
        payment_id VARCHAR(100),
        reservation_id VARCHAR(100),
        tracking_number VARCHAR(100),
        error_message TEXT,
        attempts INT DEFAULT 0,
        created_at TIMESTAMP DEFAULT NOW(),
        updated_at TIMESTAMP DEFAULT NOW()
    )
""")
cursor.execute("DELETE FROM order_state")
cursor.close()

print("✅ Database ready!")

## 📦 State Machine Approach

In [ ]:
print("📦 Order State Machine")
print("=" * 60)
print("""
Instead of losing state on crash, we persist it:

┌─────────────┐
│   CREATED   │
└──────┬──────┘
       │
       ▼
┌─────────────┐     ┌─────────────┐
│   PAYMENT   │────►│  REFUNDING  │
│   PENDING   │     └──────┬──────┘
└──────┬──────┘            │
       │                   ▼
       ▼            ┌─────────────┐
┌─────────────┐     │  CANCELLED  │
│  INVENTORY  │     └─────────────┘
│   PENDING   │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│  SHIPPING   │
│   PENDING   │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│  COMPLETED  │
└─────────────┘

After each step, we save state to database.
If we crash, we can read state and continue!
""")

In [ ]:
class OrderStatus(Enum):
    CREATED = "created"
    PAYMENT_PENDING = "payment_pending"
    INVENTORY_PENDING = "inventory_pending"
    SHIPPING_PENDING = "shipping_pending"
    COMPLETED = "completed"
    REFUNDING = "refunding"
    CANCELLED = "cancelled"
    FAILED = "failed"

class StatefulOrderProcessor:
    def __init__(self, conn):
        self.conn = conn
        self.max_retries = 3
    
    def create_order(self, order_id: str) -> dict:
        cursor = self.conn.cursor()
        cursor.execute("""
            INSERT INTO order_state (id, status, current_step)
            VALUES (%s, %s, %s)
        """, (order_id, OrderStatus.CREATED.value, 'payment'))
        cursor.close()
        return self.get_order(order_id)
    
    def get_order(self, order_id: str) -> dict:
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT id, status, current_step, payment_id, reservation_id, 
                   tracking_number, error_message, attempts
            FROM order_state WHERE id = %s
        """, (order_id,))
        row = cursor.fetchone()
        cursor.close()
        
        if not row:
            return None
        
        return {
            'id': row[0], 'status': row[1], 'current_step': row[2],
            'payment_id': row[3], 'reservation_id': row[4],
            'tracking_number': row[5], 'error_message': row[6],
            'attempts': row[7]
        }
    
    def update_order(self, order_id: str, **kwargs):
        sets = [f"{k} = %s" for k in kwargs.keys()]
        sets.append("updated_at = NOW()")
        
        cursor = self.conn.cursor()
        cursor.execute(f"""
            UPDATE order_state SET {', '.join(sets)} WHERE id = %s
        """, (*kwargs.values(), order_id))
        cursor.close()

processor = StatefulOrderProcessor(conn)
print("✅ StatefulOrderProcessor ready!")

In [ ]:
def charge_payment_with_retry(processor, order_id: str, amount: float) -> bool:
    order = processor.get_order(order_id)
    
    for attempt in range(processor.max_retries):
        try:
            print(f"      Attempt {attempt + 1}...")
            time.sleep(0.3)
            
            if random.random() < 0.3:
                raise TimeoutError("Payment timeout")
            
            payment_id = f"pay_{uuid.uuid4().hex[:8]}"
            processor.update_order(
                order_id,
                status=OrderStatus.INVENTORY_PENDING.value,
                current_step='inventory',
                payment_id=payment_id,
                attempts=attempt + 1
            )
            return True
            
        except Exception as e:
            processor.update_order(
                order_id,
                error_message=str(e),
                attempts=attempt + 1
            )
            time.sleep(2 ** attempt)
    
    processor.update_order(
        order_id,
        status=OrderStatus.FAILED.value,
        error_message="Max retries exceeded"
    )
    return False

def reserve_inventory_with_compensation(processor, order_id: str) -> bool:
    order = processor.get_order(order_id)
    
    for attempt in range(processor.max_retries):
        try:
            print(f"      Attempt {attempt + 1}...")
            time.sleep(0.2)
            
            if random.random() < 0.25:
                raise Exception("Out of stock")
            
            reservation_id = f"res_{uuid.uuid4().hex[:8]}"
            processor.update_order(
                order_id,
                status=OrderStatus.SHIPPING_PENDING.value,
                current_step='shipping',
                reservation_id=reservation_id
            )
            return True
            
        except Exception as e:
            if "Out of stock" in str(e):
                print(f"      ❌ Out of stock! Starting compensation...")
                compensate_payment(processor, order_id)
                return False
            time.sleep(2 ** attempt)
    
    compensate_payment(processor, order_id)
    return False

def compensate_payment(processor, order_id: str):
    order = processor.get_order(order_id)
    if order['payment_id']:
        print(f"      💰 Refunding payment {order['payment_id']}...")
        processor.update_order(
            order_id,
            status=OrderStatus.REFUNDING.value
        )
        time.sleep(0.3)
        
        processor.update_order(
            order_id,
            status=OrderStatus.CANCELLED.value
        )
        print(f"      ✅ Refund complete")

print("✅ Step functions with retry and compensation ready!")

In [ ]:
def process_order_stateful(order_id: str, amount: float):
    print(f"\n📦 Processing order {order_id}...")
    
    order = processor.get_order(order_id)
    if not order:
        order = processor.create_order(order_id)
    
    print(f"   Current state: {order['status']}, step: {order['current_step']}")
    
    if order['current_step'] == 'payment':
        print("   1️⃣ Charging payment...")
        if not charge_payment_with_retry(processor, order_id, amount):
            return processor.get_order(order_id)
        order = processor.get_order(order_id)
    
    if order['current_step'] == 'inventory':
        print("   2️⃣ Reserving inventory...")
        if not reserve_inventory_with_compensation(processor, order_id):
            return processor.get_order(order_id)
        order = processor.get_order(order_id)
    
    if order['current_step'] == 'shipping':
        print("   3️⃣ Creating shipping label...")
        time.sleep(0.2)
        tracking = f"1Z{uuid.uuid4().hex[:12].upper()}"
        processor.update_order(
            order_id,
            status=OrderStatus.COMPLETED.value,
            current_step='done',
            tracking_number=tracking
        )
    
    return processor.get_order(order_id)

print("🔥 Testing stateful order processing...")
print("=" * 60)

test_order_id = str(uuid.uuid4())
result = process_order_stateful(test_order_id, 99.99)

print(f"\n📊 Final state:")
print(f"   Status: {result['status']}")
print(f"   Payment ID: {result['payment_id']}")
print(f"   Reservation ID: {result['reservation_id']}")
print(f"   Tracking: {result['tracking_number']}")

## 💥 Simulating a Crash

In [ ]:
print("💥 Simulating Server Crash")
print("=" * 60)

crash_order_id = str(uuid.uuid4())

print("\n1️⃣ Starting order, will 'crash' after payment...")
processor.create_order(crash_order_id)
processor.update_order(
    crash_order_id,
    status=OrderStatus.INVENTORY_PENDING.value,
    current_step='inventory',
    payment_id='pay_crashed123'
)
print(f"   Order saved to DB, then server CRASHES! 💥")

print("\n2️⃣ Server restarts... reading state from DB...")
recovered_order = processor.get_order(crash_order_id)
print(f"   Found order: {recovered_order['id'][:8]}...")
print(f"   Status: {recovered_order['status']}")
print(f"   Current step: {recovered_order['current_step']}")
print(f"   Payment already done: {recovered_order['payment_id']}")

print("\n3️⃣ Resuming from where we left off...")
result = process_order_stateful(crash_order_id, 99.99)

print(f"\n✅ Order recovered and completed!")
print(f"   Final status: {result['status']}")

## 😰 Why This Gets Messy

In [ ]:
print("😰 Problems with Manual State Management")
print("=" * 60)
print("""
What we had to build manually:

1. STATE PERSISTENCE
   ─────────────────────────────────────────────────────────
   - Create database schema
   - Save after every step
   - Handle concurrent updates
   - What if DB write fails?

2. RETRY LOGIC
   ─────────────────────────────────────────────────────────
   - Track attempt count
   - Exponential backoff
   - Max retry limits
   - Different retry policies per step?

3. COMPENSATION (ROLLBACK)
   ─────────────────────────────────────────────────────────
   - Define undo for each step
   - Handle compensation failures
   - Compensation ordering
   - Partial compensation?

4. ERROR HANDLING
   ─────────────────────────────────────────────────────────
   - Transient vs permanent errors
   - Timeout handling
   - Error propagation
   - Dead letter queue?

5. RECOVERY
   ─────────────────────────────────────────────────────────
   - Find stuck orders
   - Resume from correct step
   - Handle version changes
   - Multiple workers?

THIS IS A LOT OF CODE!
And we haven't even handled:
- Timeouts (waiting for webhooks)
- Human tasks (manual approval)
- Parallel steps
- Workflow versioning
""")

In [ ]:
print("📊 Lines of Code Comparison")
print("=" * 60)
print("""
                    Naive      With State    Temporal
                    ─────      ──────────    ────────
Business Logic:      20           20            20
State Management:     0           50             0
Retry Logic:          0           30             0
Compensation:         0           40             5
Error Handling:       5           60             0
Recovery:             0           40             0
                    ─────      ──────────    ────────
TOTAL:               25          240            25

With Temporal, we write business logic.
Temporal handles the rest!
""")

## 🧪 Quick Quiz

1. **What's the purpose of saving state after each step?**

2. **What is a compensation action?**

3. **Why does manual state management become unmaintainable?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Saving state after each step:")
print("   - Enables recovery after crash")
print("   - Know which steps completed")
print("   - Resume from correct position")
print()
print("2. Compensation action:")
print("   - Undo a completed step")
print("   - Example: refund after payment")
print("   - Used when later steps fail")
print()
print("3. Why it becomes unmaintainable:")
print("   - Too much infrastructure code")
print("   - Business logic hidden")
print("   - Every edge case needs handling")
print("   - Hard to modify workflow")

## 📚 Summary

### What We Built

1. **State machine** with database persistence
2. **Retry logic** with exponential backoff
3. **Compensation** for rollbacks
4. **Crash recovery** from saved state

### Problems

1. Too much boilerplate code
2. Business logic buried in infrastructure
3. Hard to add new steps
4. Doesn't handle long waits well

### Next Up

In **Notebook 3**, we'll explore event sourcing:
- Using event logs for state
- Decoupling with events
- Better but still complex